In [ ]:
import pandas as pd
import requests
import time

base_url = "https://api.themoviedb.org/3/movie/top_rated"

headers = {
    "accept": "application/json",
    "Authorization": "Bearer YOUR_ACCESS_TOKEN"
}

all_data = []

session = requests.Session()   # reuse connection (important)

# First call
response = session.get(base_url, headers=headers, params={"language": "en-US", "page": 1})
data = response.json()
total_pages = data["total_pages"]

print("Total Pages:", total_pages)

for page in range(1, total_pages + 1):

    try:
        print(f"Fetching page {page}...")

        response = session.get(
            base_url,
            headers=headers,
            params={"language": "en-US", "page": page},
            timeout=10
        )

        if response.status_code == 200:
            results = response.json().get("results", [])

            if results:
                temp_df = pd.DataFrame(results)[
                    ['id', 'title', 'overview', 'release_date', 'vote_count', 'vote_average']
                ]
                all_data.append(temp_df)

        else:
            print("Status error:", response.status_code)

        time.sleep(0.5)   # IMPORTANT: slow down requests

    except requests.exceptions.RequestException as e:
        print("Error on page", page, ":", e)
        print("Sleeping 5 seconds...")
        time.sleep(5)
        continue

df = pd.concat(all_data, ignore_index=True)

print("Final Shape:", df.shape)


## 🔴 What Does Status Error 400 Mean?

HTTP 400 = Bad Request

In our case:

we are looping until page 534

But TMDB API has a rule:

❗ Maximum accessible pages = 500

Even if total_pages = 534

TMDB only allows you to fetch up to page 500.

So when our loop hits:

page 501+


Server says:

400 Bad Request


That’s why you see:

Fetching page 518...
Status error: 400

In [17]:
#temp_df = pd.DataFrame(response.json()['results'])[['id','title','overview','release_date','vote_count','vote_average']]

In [27]:
df.shape

(10000, 6)

In [28]:
df.head()

,id,title,overview,release_date,vote_count,vote_average
0,278,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,1994-09-23,29736,8.715
1,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",1972-03-14,22461,8.687
2,240,The Godfather Part II,In the continuing saga of the Corleone crime f...,1974-12-20,13591,8.600
3,424,Schindler's List,The true story of how businessman Oskar Schind...,1993-12-15,17111,8.567
4,389,12 Angry Men,The defense and the prosecution have rested an...,1957-04-10,9743,8.555


In [29]:
df.to_csv("tmdb_top_rated_movies.csv", index=False, encoding="utf-8")


In [32]:
# Check FilePath where saved
import os
os.getcwd()


'C:\\Users\\shrav'

In [31]:
df.shape

(10000, 6)